# BERT-Tiny Indic LID — Model Comparison

Compare prediction exports on aligned examples. This notebook computes coverage, operational accuracy (unsupported predictions count as incorrect), supported-prediction accuracy, macro/weighted F1, per-language metrics, confusion matrices, and recorded latency percentiles.

**Important:** results are only valid if the reference label is trustworthy and all model predictions refer to the same text rows. Timing comparisons are not fair unless units, hardware, warm-up, batch size, and network/API inclusion are documented. This notebook does not run the models; it analyzes prediction exports.

## 1. Configure input files and columns

Put your prediction exports in `data/`. Set the column names below to match each CSV. Use one row per shared evaluation example and a stable `id` column. For the Sarvam subset, run a separate paired comparison because it may not contain the full evaluation set.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support)

DATA_DIR = Path('data')
OUT_DIR = Path('results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
LABELS = ['as-IN','bn-IN','en-IN','gu-IN','hi-IN','kn-IN','ml-IN','mr-IN','pa-IN','ta-IN','te-IN','none']

# Edit filenames and columns to match your exported prediction sheets.
FILES = {
    'BERT-Tiny E4': {'file': 'bert_predictions.csv', 'id': 'eval_id', 'text': 'text', 'truth': 'label', 'pred': 'bert_pred', 'latency': 'bert_latency_ms'},
    'fast-langdetect': {'file': 'fastlangdetect_predictions.csv', 'id': 'eval_id', 'text': 'text', 'truth': 'label', 'pred': 'fastlangdetect_pred', 'latency': 'latency_ms'},
    'MIRA-IndicLID': {'file': 'mira_predictions.csv', 'id': 'eval_id', 'text': 'text', 'truth': 'label', 'pred': 'pred_MIRA_IndicLID', 'latency': 'latency_ms_MIRA_IndicLID'},
}

# Add Sarvam here only if its export is on the same rows; otherwise analyze it in a separate subset run.
print('Input directory:', DATA_DIR.resolve())
print('Expected files:', [v['file'] for v in FILES.values()])

In [ ]:
def norm_label(x):
    if pd.isna(x): return '__unsupported__'
    s = str(x).strip()
    aliases = {'hi':'hi-IN','en':'en-IN','bn':'bn-IN','as':'as-IN','gu':'gu-IN',
               'kn':'kn-IN','ml':'ml-IN','mr':'mr-IN','pa':'pa-IN','ta':'ta-IN','te':'te-IN',
               'unknown':'none','other':'none','__unsupported__':'__unsupported__'}
    s = aliases.get(s, s)
    return s if s in LABELS else '__unsupported__'

def load_export(model_name, cfg):
    path = DATA_DIR / cfg['file']
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Put the export in data/ and update FILES.')
    df = pd.read_csv(path)
    needed = [cfg['id'], cfg['text'], cfg['truth'], cfg['pred']]
    missing = [c for c in needed if c not in df.columns]
    if missing: raise KeyError(f'{model_name}: missing columns {missing}; available: {list(df.columns)}')
    out = df[needed + ([cfg['latency']] if cfg.get('latency') in df.columns else [])].copy()
    out = out.rename(columns={cfg['id']:'id',cfg['text']:'text',cfg['truth']:'truth',cfg['pred']:'pred'})
    if cfg.get('latency') in out.columns: out = out.rename(columns={cfg['latency']:'latency'})
    out['truth'] = out['truth'].map(norm_label)
    out['pred'] = out['pred'].map(norm_label)
    out['model'] = model_name
    return out

exports = {name: load_export(name, cfg) for name, cfg in FILES.items()}
for name, df in exports.items():
    print(f'{name}: rows={len(df):,}, unique IDs={df.id.nunique():,}, duplicate IDs={df.id.duplicated().sum():,}')

## 2. Validate row alignment
Never compare model scores on different examples without explicitly reporting the subset.

In [ ]:
base_name = next(iter(exports))
base = exports[base_name][['id','text','truth']].drop_duplicates('id').sort_values('id').reset_index(drop=True)
alignment = []
for name, df in exports.items():
    d = df[['id','text','truth']].drop_duplicates('id')
    m = base.merge(d, on='id', how='outer', suffixes=('_base','_model'), indicator=True)
    both = m[m['_merge']=='both']
    alignment.append({'model':name,'rows':len(df),'unique_ids':df.id.nunique(),
        'missing_vs_base':int((m['_merge']=='left_only').sum()),'extra_vs_base':int((m['_merge']=='right_only').sum()),
        'text_mismatches':int((both.text_base != both.text_model).sum()),
        'label_mismatches':int((both.truth_base != both.truth_model).sum())})
alignment_df = pd.DataFrame(alignment)
display(alignment_df)
alignment_df.to_csv(OUT_DIR/'alignment_audit.csv', index=False)
if (alignment_df[['missing_vs_base','extra_vs_base','text_mismatches','label_mismatches']].to_numpy()!=0).any():
    warnings.warn('Exports are not fully aligned. Restrict metrics to verified shared IDs and inspect mismatches.')

## 3. Metrics on the common, verified subset

In [ ]:
common = set.intersection(*[set(df.id.dropna()) for df in exports.values()])
master = base[base.id.isin(common)].copy()
for name, df in exports.items():
    part = df[['id','text','truth','pred'] + (['latency'] if 'latency' in df.columns else [])].drop_duplicates('id')
    master = master.merge(part[['id','pred'] + (['latency'] if 'latency' in part.columns else [])].rename(columns={'pred':f'{name}__pred', 'latency':f'{name}__latency'}),on='id',how='inner')
print('Common rows:', len(master))

def score_model(y, pred, name):
    y = pd.Series(y).astype(str); pred = pd.Series(pred).astype(str)
    supported = pred.isin(LABELS)
    # Since norm_label maps unsupported outputs to a sentinel, coverage is predictions mapped to target labels.
    coverage = float(supported.mean()) if len(pred) else np.nan
    operational = (pred == y).mean() if len(y) else np.nan
    ys, ps = y[supported], pred[supported]
    sup_acc = accuracy_score(ys, ps) if len(ys) else np.nan
    p,r,f,_ = precision_recall_fscore_support(y, pred, labels=LABELS, average=None, zero_division=0)
    macro = precision_recall_fscore_support(y, pred, labels=LABELS, average='macro', zero_division=0)[2]
    weighted = precision_recall_fscore_support(y, pred, labels=LABELS, average='weighted', zero_division=0)[2]
    return {'model':name,'n':len(y),'coverage':coverage,'operational_accuracy':operational,
            'supported_prediction_accuracy':sup_acc,'balanced_accuracy':balanced_accuracy_score(y,pred),
            'macro_f1':macro,'weighted_f1':weighted}

metric_rows=[]
for name in exports:
    metric_rows.append(score_model(master.truth, master[f'{name}__pred'], name))
metrics = pd.DataFrame(metric_rows).sort_values('macro_f1',ascending=False)
display(metrics)
metrics.to_csv(OUT_DIR/'model_metrics.csv',index=False)

In [ ]:
ax = metrics.set_index('model')[['operational_accuracy','macro_f1','weighted_f1']].plot(kind='bar',figsize=(11,5),ylim=(0,1))
ax.set_title('Model quality on common verified rows'); ax.set_ylabel('Score'); ax.set_xlabel(''); plt.xticks(rotation=20,ha='right'); plt.tight_layout()
plt.savefig(OUT_DIR/'quality_metrics.png',dpi=160); plt.show()

## 4. Per-language scores and confusion matrices

In [ ]:
per_lang=[]
for name in exports:
    rep = classification_report(master.truth, master[f'{name}__pred'], labels=LABELS, output_dict=True, zero_division=0)
    for label in LABELS:
        row = rep.get(label,{})
        per_lang.append({'model':name,'label':label,'precision':row.get('precision',0),
                         'recall':row.get('recall',0),'f1':row.get('f1-score',0),'support':row.get('support',0)})
per_language = pd.DataFrame(per_lang)
display(per_language.head(24))
per_language.to_csv(OUT_DIR/'per_language_metrics.csv',index=False)

for name in exports:
    cm = confusion_matrix(master.truth, master[f'{name}__pred'], labels=LABELS)
    cm_df = pd.DataFrame(cm,index=LABELS,columns=LABELS)
    safe = name.lower().replace(' ','_').replace('/','_')
    cm_df.to_csv(OUT_DIR/f'confusion_matrix_{safe}.csv')
    fig, ax = plt.subplots(figsize=(9,8))
    ConfusionMatrixDisplay(cm,display_labels=LABELS).plot(ax=ax,xticks_rotation=90,colorbar=False,values_format='d')
    ax.set_title(f'Confusion matrix — {name}'); fig.tight_layout(); fig.savefig(OUT_DIR/f'confusion_matrix_{safe}.png',dpi=160); plt.show()

## 5. Recorded latency audit
This section summarizes the latency values already present in the export. It does not re-measure inference. Confirm units before interpreting the results.

In [ ]:
lat_rows=[]
for name in exports:
    col=f'{name}__latency'
    if col not in master.columns: continue
    vals=pd.to_numeric(master[col],errors='coerce').dropna()
    if vals.empty: continue
    lat_rows.append({'model':name,'n_timing_values':len(vals),'mean':vals.mean(),'p50':vals.quantile(.50),
                     'p90':vals.quantile(.90),'p95':vals.quantile(.95),'p99':vals.quantile(.99),'max':vals.max()})
latency = pd.DataFrame(lat_rows)
display(latency)
latency.to_csv(OUT_DIR/'recorded_latency_summary.csv',index=False)
if len(latency):
    ax=latency.set_index('model')[['p50','p90','p95']].plot(kind='bar',figsize=(10,5))
    ax.set_title('Recorded latency percentiles — verify units'); ax.set_ylabel('Recorded value (unit not assumed)'); ax.set_xlabel('')
    plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.savefig(OUT_DIR/'recorded_latency_percentiles.png',dpi=160); plt.show()

## 6. Export and reporting checklist
Before publishing results, verify: (1) held-out ground truth is independent and correctly labeled; (2) all models are scored on identical IDs/texts; (3) unsupported predictions are handled consistently; (4) each class has adequate support; (5) the label mapping is documented; (6) latency units and protocol are known; (7) local model latency and API/network latency are separated; and (8) dataset and model licenses permit redistribution.